##### 🌾 CSIRO Biomass Prediction - Two-Stream ConvNeXt Inference

## Reference:https://www.kaggle.com/code/none00000/lb-0-57-infer-model-code

## 📊 Performance
- **Public LB Score:** 0.58
- **Local CV Score:** 0.XXX ± 0.XXX   ※wait for training notebook!
- **Model:** ConvNeXt-Tiny with Two-Stream Architecture
- **Ensemble:** 5-Fold + 3-View TTA

---

## 🎯 What's in This Notebook?

This inference notebook implements a **two-stream architecture** for predicting pasture biomass from top-view images. Key features:

- ✅ **Two-Stream CNN**: Processes left/right image halves independently
- ✅ **Three-Head Regression**: Dedicated heads for Total, GDM, and Green biomass
- ✅ **5-Fold Ensemble**: Robust predictions through cross-validation
- ✅ **Test-Time Augmentation**: Original + HFlip + VFlip (3 views)
- ✅ **Clean & Documented Code**: Easy to understand and modify

---

## 💬 Feedback & Discussion

If you find this notebook helpful:
- 👍 **Please upvote** to support my work
- 💬 **Leave a comment** with your questions or suggestions
- 🔔 **Follow me** for more competitions and insights

**Questions? Issues?** Feel free to comment below, and I'll respond ASAP!



In [ ]:
"""
CSIRO Biomass Competition - Inference Pipeline (TTA + Ridge)
================================================================================
This script performs predictions on test data using a trained Ridge model.

Pipeline Overview:
1. Test Data Preparation: Load CSV → Extract unique images
2. Model Loading: Load Ridge model
3. TTA Inference: 5 Views × Ridge prediction
4. Post-processing: Average TTA predictions
5. Submission Creation: Wide format → Long format conversion
"""

from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
from collections import OrderedDict
import os
import gc
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import cv2
from tqdm import tqdm

# ============================================================================
# Configuration Management
# ============================================================================

@dataclass
class InferenceConfig:
    """
    Data class for managing inference pipeline configuration.
    
    Items that must match training configuration:
    - model_name
    - img_size
    - target column names
    """
    
    # Path settings
    base_path: Path = Path('/kaggle/input/csiro-biomass')
    test_csv: Path = field(init=False)
    test_image_dir: Path = field(init=False)
    backbone_dir: Path = Path('/kaggle/input/vit-large-patch16-dinov3/vit_large_patch16_dinov3.pth')
    submission_file: str = 'submission.csv'
    ridge_model_path: Path = Path('/kaggle/input/ridge-models/ridge_seed_13.joblib')
    
    # Model settings (must match training)
    model_name: str = 'vit_large_patch16_dinov3'
    img_size: int = 1008
    
    # Device settings
    device: torch.device = field(default_factory=lambda: torch.device(
        'cuda:0' if torch.cuda.is_available() else 'cpu'
    ))
    
    # Inference settings
    batch_size: int = 16
    num_workers: int = 4
    
    # Target settings (must match training)
    train_target_cols: list[str] = field(default_factory=lambda: [
        'Dry_Total_g', 'GDM_g', 'Dry_Green_g'
    ])
    
    all_target_cols: list[str] = field(default_factory=lambda: [
        'Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g'
    ])
    
    def __post_init__(self) -> None:
        """Construct paths after initialization"""
        self.test_csv = self.base_path / 'test.csv'
        self.test_image_dir = self.base_path / 'test'
    
    def display_info(self) -> None:
        """Display configuration information"""
        print(f"{'='*70}")
        print(f"Inference Configuration")
        print(f"{'='*70}")
        print(f"Device: {self.device}")
        print(f"Backbone: {self.model_name}")
        print(f"Image Size: {self.img_size}x{self.img_size}")
        print(f"Batch Size: {self.batch_size}")
        print(f"Ridge Model: {self.ridge_model_path}")
        print(f"TTA: Multiple Views (Original, HFlip, VFlip, Rot180, Transpose)")
        print(f"{'='*70}\n")


# ============================================================================
# TTA Augmentation
# ============================================================================

class TTATransformFactory:
    """
    Factory class for generating Test Time Augmentation transforms.
    
    Provides 3 different views:
    1. Original (no augmentation)
    2. Horizontal flip
    3. Vertical flip
    """
    
    def __init__(self, img_size: int):
        """
        Args:
            img_size: Image size after resizing
        """
        self.img_size = img_size
        
        # Base transforms common to all views
        self.base_transforms = [
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ]
    
    def get_tta_transforms(self) -> list[A.Compose]:
        """
        Generate 3 transform pipelines for TTA.
        
        Returns:
            List of 3 Albumentations Compose objects
            
        Why not: Not adding more TTA variations
            → Considering trade-off with inference time
        """
        # View 1: Original
        original = A.Compose([
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])
        
        # View 2: Horizontal flip
        hflip = A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])
        
        # View 3: Vertical flip
        vflip = A.Compose([
            A.VerticalFlip(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])

        rot180 = A.Compose([
            A.HorizontalFlip(p=1.0),
            A.VerticalFlip(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])

        transp = A.Compose([
            A.Transpose(p=1.0),
            A.Resize(self.img_size, self.img_size),
            *self.base_transforms
        ])
        
        return [original, hflip, vflip, rot180, transp]


# ============================================================================
# Dataset
# ============================================================================

class TestBiomassDataset(Dataset):
    """
    Two-stream dataset for testing.
    
    Accepts a specific transform pipeline for TTA and applies
    the same augmentation to both left and right images.
    
    Returns:
        tuple: (img_left, img_right)
    """
    
    def __init__(
        self,
        df: pd.DataFrame,
        transform_pipeline: A.Compose,
        image_dir: Path
    ):
        """
        Args:
            df: DataFrame containing image paths
            transform_pipeline: Augmentation pipeline to apply
            image_dir: Image directory path
        """
        self.df = df
        self.transform = transform_pipeline
        self.image_dir = image_dir
        self.image_paths = df['image_path'].values
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Get one sample.
        
        Args:
            idx: Sample index
            
        Returns:
            (left_image, right_image)
            
        Why not: Not applying different augmentations to left/right as in training
            → During TTA, apply same transform to both images to preserve symmetry
        """
        img_path = self.image_paths[idx]
        full_path = self.image_dir / Path(img_path).name
        
        # Load image (return black image on error)
        image = cv2.imread(str(full_path))
        
        if image is None:
            print(f"Warning: Failed to load image: {full_path} → Returning black image")
            image = np.zeros((1000, 2000, 3), dtype=np.uint8)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Split into left and right
        height, width = image.shape[:2]
        mid_point = width // 2
        img_left = image[:, :mid_point]
        img_right = image[:, mid_point:]
        
        # Apply same transform to both
        img_left_tensor = self.transform(image=img_left)['image']
        img_right_tensor = self.transform(image=img_right)['image']
        
        return img_left_tensor, img_right_tensor


# ============================================================================
# Model
# ============================================================================
class BackboneModel(nn.Module):
    """
    Two-stream feature extractor.
    
    Processes left/right halves and returns concatenated features.
    """
    
    def __init__(self, model_name: str, pretrained: bool = False):
        """
        Args:
            model_name: timm model name
            pretrained: Always False (weights loaded separately)
        """
        super().__init__()
        # Shared backbone
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
        )
        nf = self.backbone.num_features
        image_feature_dim = nf * 2
    
    def forward(
        self,
        img_left: torch.Tensor,
        img_right: torch.Tensor,
    ) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            img_left: Left image [B, C, H, W]
            img_right: Right image [B, C, H, W]
            
        Returns:
            Concatenated features [B, 2048]
        """
        fl = self.backbone(img_left)
        fr = self.backbone(img_right)
        image_features = torch.cat([fl, fr], dim=1)
        return image_features


# ============================================================================
# Model Loader
# ============================================================================

class ModelLoader:
    """
    Class for loading trained models.
    
    Handles weights saved with DataParallel.
    """
    
    def __init__(self, config: InferenceConfig):
        """
        Args:
            config: Configuration object
        """
        self.config = config
        
    def load_model(self, path) -> nn.Module:
        print(f"\nLoading backbone from {path}")
        
        model_path = path
        
        if not model_path.exists():
            raise FileNotFoundError(f"Model file not found: {model_path}")
        
        # Initialize model
        backbone_model = BackboneModel(self.config.model_name, pretrained=False)
        
        # Load weights
        state_dict = torch.load(model_path, map_location=self.config.device)
        
        # Handle DataParallel (remove 'module.' prefix)
        state_dict = self._remove_dataparallel_prefix(state_dict)
        
        backbone_model.backbone.load_state_dict(state_dict)
        backbone_model.eval()  # Evaluation mode
        backbone_model.to(self.config.device)

        if torch.cuda.device_count() > 1:
            print(f"   -> ⚡ Using {torch.cuda.device_count()} GPUs for Backbone")
            backbone_model.to('cuda:0') 
            backbone_model = nn.DataParallel(backbone_model)
        
        print(f"✓ Successfully loaded backbone\n")
        return backbone_model
    
    def load_ridge_model(self) -> object:
        """
        Loads a single Ridge model from the configured path.
        
        The model is a sklearn MultiOutputRegressor predicting 5 targets:
        [Dry_Green_g, Dry_Dead_g, Dry_Clover_g, GDM_g, Dry_Total_g].
        
        Returns:
            Trained Ridge model (MultiOutputRegressor)
        """
        path = self.config.ridge_model_path
        print(f"Loading Ridge model from {path}...")
        
        if not path.exists():
            raise FileNotFoundError(f"Ridge model not found: {path}")
        
        model = joblib.load(path)
        print(f"  ✅ Loaded Ridge model\n")
        return model
    
    @staticmethod
    def _remove_dataparallel_prefix(state_dict: dict) -> dict:
        """
        Remove 'module.' prefix from DataParallel-saved weights.
        
        Args:
            state_dict: Model weight dictionary
            
        Returns:
            Weight dictionary with prefix removed
            
        Why not: Not using try-except with direct load_state_dict
            → Explicitly handling prefix presence improves readability
        """
        if not any(k.startswith('module.') for k in state_dict.keys()):
            return state_dict  # No prefix
        
        new_state_dict = OrderedDict()
        for key, value in state_dict.items():
            new_key = key.replace('module.', '')
            new_state_dict[new_key] = value
        
        return new_state_dict


# ============================================================================
# Inference Engines
# ============================================================================

class GlobalFeatureExtractor:
    def __init__(self, backbone, config):
        self.backbone = backbone
        self.config = config
        self.device = config.device
        self.backbone.to(self.device)
        self.backbone.eval()

    def get_all_embeddings(self, test_df, tta_transforms):
        """
        Runs backbone on ALL TTA variants.
        
        Returns:
            X_massive (np.ndarray): Shape (N_images * K_transforms, 2048)
            N (int): Number of original images
            K (int): Number of transforms
        """
        N = len(test_df)
        K = len(tta_transforms)
        all_folds_features = []

        print(f"--- 🚀 Starting Global Feature Extraction ({K} views) ---")
        
        # 1. Loop through each TTA Transform
        for i, transform in enumerate(tta_transforms):
            print(f"  > Processing View {i+1}/{K}...")
            
            # Setup Loader for this specific view
            dataset = TestBiomassDataset(
                test_df,
                transform,
                self.config.test_image_dir
            )
            
            loader = DataLoader(
                dataset,
                batch_size=self.config.batch_size,
                shuffle=False,
                num_workers=self.config.num_workers,
                pin_memory=True
            )
            
            # Extract features for this view
            view_feats = self._extract_one_view(loader)
            all_folds_features.append(view_feats)

        # 2. Stack Vertically
        # Result: [View1_Img1, View1_Img2... | View2_Img1, View2_Img2...]
        X_massive = np.vstack(all_folds_features)
        
        expected_rows = N * K
        assert X_massive.shape[0] == expected_rows, f"Shape mismatch! Got {X_massive.shape}, expected ({expected_rows}, ...)"
        
        print(f"--- ✅ Extraction Complete. Matrix Shape: {X_massive.shape} ---")
        return X_massive, N, K

    def _extract_one_view(self, loader):
        """Helper to run backbone on one loader"""
        feats_list = []
        with torch.inference_mode():
            for img_left, img_right in tqdm(loader, leave=False):
                img_left = img_left.to(self.device)
                img_right = img_right.to(self.device)
                
                with torch.amp.autocast('cuda'):
                    # Ensure your backbone returns (Batch, 2048)
                    batch_feats = self.backbone(img_left, img_right)
                
                feats_list.append(batch_feats.cpu().numpy())
        return np.vstack(feats_list)

class RidgePredictor:
    """
    Predicts using a single Ridge model with TTA averaging.
    """
    def __init__(self, ridge_model, config):
        self.ridge_model = ridge_model
        self.config = config

    def predict_all(self, X_massive, N, K):
        """
        Predicts on the massive matrix and averages predictions back to N images.
        """
        print("--- 🧠 Running Ridge Prediction ---")
        
        # Ridge prediction on all TTA views
        print("  > Ridge Model...")
        ridge_preds = self._predict_ridge(X_massive)
        ridge_final = self._reshape_and_avg(ridge_preds, N, K)
        
        return ridge_final

    def _predict_ridge(self, X):
        """
        Runs inference using a single Ridge model.
        The model is a MultiOutputRegressor predicting 5 targets in order:
        [Dry_Green_g, Dry_Dead_g, Dry_Clover_g, GDM_g, Dry_Total_g].
        """
        # Predict on all (N*K) samples at once
        preds = self.ridge_model.predict(X)
        
        # Unpack into dictionary
        # Order from MultiOutputRegressor: [Green, Dead, Clover, GDM, Total]
        result = {}
        if preds.ndim == 1:
            result['total'] = preds
        elif preds.shape[1] == 5:
            result['green']  = preds[:, 0]
            result['dead']   = preds[:, 1]
            result['clover'] = preds[:, 2]
            result['gdm']    = preds[:, 3]
            result['total']  = preds[:, 4]
        else:
            raise ValueError(f"Unexpected prediction shape: {preds.shape}")
        
        return result

    def _reshape_and_avg(self, preds_dict, N, K):
        """
        Collapses the TTA dimension.
        Input: Dict of arrays (N*K,)
        Output: Dict of arrays (N,)
        """
        final = {}
        for key, arr in preds_dict.items():
            # 1. Split into K chunks of size N
            chunks = np.split(arr, K)
            # 2. Stack and Mean
            stacked = np.column_stack(chunks)
            final[key] = np.mean(stacked, axis=1)
            
        return final

# ============================================================================
# Submission Creation
# ============================================================================

class SubmissionCreator:
    """
    Class for creating Kaggle submission CSV from predictions.
    """
    
    def __init__(self, config: InferenceConfig):
        """
        Args:
            config: Configuration object
        """
        self.config = config
    
    def create(
        self,
        predictions: dict[str, np.ndarray],
        test_df_long: pd.DataFrame,
        test_df_unique: pd.DataFrame
    ) -> None:
        """
        Create and save submission CSV from predictions.
        
        Args:
            predictions: {'total': [N], 'gdm': [N], 'green': [N], 'clover': [N], 'dead': [N]}
            test_df_long: Original test.csv (long format)
            test_df_unique: DataFrame of unique images
            
        Processing flow:
        1. Calculate 5 targets from 3 predictions
        2. Create wide format DataFrame
        3. Convert to long format (melt)
        4. Merge with sample_id
        5. Save CSV
        """
        print("Creating submission CSV...")
        
        # 1. Get predictions
        pred_total = predictions['total']
        pred_gdm = predictions['gdm']
        pred_green = predictions['green']
        pred_clover = predictions['clover']
        pred_dead = predictions['dead']
        
        # 3. Create wide format DataFrame
        preds_wide = pd.DataFrame({
            'image_path': test_df_unique['image_path'],
            'Dry_Green_g': pred_green,
            'Dry_Dead_g': pred_dead,
            'Dry_Clover_g': pred_clover,
            'GDM_g': pred_gdm,
            'Dry_Total_g': pred_total
        })
        
        # 4. Convert to long format (unpivot)
        preds_long = preds_wide.melt(
            id_vars=['image_path'],
            value_vars=self.config.all_target_cols,
            var_name='target_name',
            value_name='target'
        )
        
        # 5. Merge with original test.csv to get sample_id
        submission = pd.merge(
            test_df_long[['sample_id', 'image_path', 'target_name']],
            preds_long,
            on=['image_path', 'target_name'],
            how='left'
        )
        
        # 6. Format and save
        submission = submission[['sample_id', 'target']]
        submission.to_csv(self.config.submission_file, index=False)
        
        print(f"\n🎉 Submission saved: {self.config.submission_file}")
        print("\n--- First 5 rows ---")
        print(submission.head())
        print("\n--- Last 5 rows ---")
        print(submission.tail())


# ============================================================================
# Inference Pipeline
# ============================================================================
class InferencePipeline:
    """
    Class that orchestrates the entire inference pipeline.
    """
    
    def __init__(self, config: InferenceConfig):
        """
        Args:
            config: Configuration object
        """
        self.config = config
        self.model_loader = ModelLoader(config)
        self.tta_factory = TTATransformFactory(config.img_size)
        self.submission_creator = SubmissionCreator(config)
    
    def run(self) -> None:
        """
        Execute the entire inference pipeline.
        
        Processing flow:
        1. Load test data
        2. Load backbone feature extractor
        3. Load Ridge model
        4. TTA inference + Ridge prediction
        5. Post-process and clamp
        6. Create submission
        """
        print(f"\n{'='*70}")
        print(f"🚀 Starting Inference Pipeline")
        print(f"{'='*70}")
        
        try:
            # 1. Load test data
            test_df_long, test_df_unique = self._load_test_data()
            
            # 2. Load backbone feature extractor
            backbone = self.model_loader.load_model(self.config.backbone_dir)
            feat_extractor = GlobalFeatureExtractor(backbone, self.config)
            del backbone
            torch.cuda.empty_cache()
            
            # 3. Load Ridge model (single seed)
            ridge_model = self.model_loader.load_ridge_model()
            
            # 4. TTA: extract features from all views
            tta_transforms = self.tta_factory.get_tta_transforms()
            X_massive, N, K = feat_extractor.get_all_embeddings(test_df_unique, tta_transforms)
            del feat_extractor
            torch.cuda.empty_cache()
            
            # 5. Predict with Ridge model + TTA averaging
            predictor = RidgePredictor(ridge_model, self.config)
            preds_ridge = predictor.predict_all(X_massive, N, K)
            
            # 6. Final clamp (physics check)
            final_preds = {}
            for key in preds_ridge.keys():
                final_preds[key] = np.maximum(preds_ridge[key], 0)

            # 7. Create submission
            self.submission_creator.create(
                final_preds,
                test_df_long,
                test_df_unique
            )
            
            print("\n✨ Inference Pipeline Completed ✨")
            
        except Exception as e:
            print(f"\n❌ Error occurred: {e}")
            raise
        
        finally:
            # Free memory
            gc.collect()
            torch.cuda.empty_cache()
    
    def _load_test_data(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        """
        Load test data.
        
        Returns:
            (test_df_long, test_df_unique)
            - test_df_long: Original long format (with sample_id)
            - test_df_unique: Unique images only
            
        Raises:
            FileNotFoundError: If test.csv not found
        """
        print(f"\nLoading test data: {self.config.test_csv}")
        
        if not self.config.test_csv.exists():
            raise FileNotFoundError(f"test.csv not found: {self.config.test_csv}")
        
        test_df_long = pd.read_csv(self.config.test_csv)
        test_df_unique = test_df_long.drop_duplicates(
            subset=['image_path']
        ).reset_index(drop=True)
        
        print(f"  Long format: {len(test_df_long)} rows")
        print(f"  Unique images: {len(test_df_unique)} images\n")
        
        return test_df_long, test_df_unique


# ============================================================================
# Main Execution
# ============================================================================

if __name__ == '__main__':
    # Initialize configuration
    config = InferenceConfig()
    config.display_info()
    
    # Run pipeline
    pipeline = InferencePipeline(config)
    pipeline.run()
    
    print("\n" + "="*70)
    print("🎊 Inference Pipeline Completed!")
    print("="*70)

In [ ]:
0  ID1001187975__Dry_Clover_g   0.167680
1    ID1001187975__Dry_Dead_g  37.301017
2   ID1001187975__Dry_Green_g  29.283626
3   ID1001187975__Dry_Total_g  68.008954
4         ID1001187975__GDM_g  30.707930